# DKTC v5 - Score 최적화 최종 버전

## 핵심 문제 진단
v2에서 일반 대화를 19/500개만 예측 (3.8%) → 실제 test에서 ~378/500 (75.6%)
- 원인 1: 랜덤 문장 3~5개 이어붙이기 → 부자연스러운 합성 데이터
- 원인 2: train 일반:위험 = 1000:3950 (20%) → test 75% 일반 → 분포 불일치
- 원인 3: 모델이 "합성 vs 실제" 패턴을 학습함

## v5 새로운 전략 (5개)
- [v5-1] Natural Individual Sampling: 문장 이어붙이기 대신 개별 문장 사용
- [v5-2] 3x Normal Scale-up: 합성 일반대화 1000개 → 3000+개
- [v5-3] Prior Shift Calibration: 예상 test 분포로 사후 확률 보정
- [v5-4] Confidence-based Normal Fallback: 위험 확률 낮으면 일반으로 분류
- [v5-5] Template Augmentation: 100개 수작업 일상 대화 템플릿

## 유지 전략 (v3/v4)
- K-Fold (5-Fold) + Multi-Model (KcELECTRA + KcBERT) + R-Drop + Focal Loss
- LLRD + FGM + EMA + Label Smoothing + Weighted Ensemble + Pseudo Labeling

In [ ]:
# ============================================================
# STEP 0-1: 한국어 폰트 설치 + 런타임 재시작
# ============================================================
!apt-get install -y fonts-nanum > /dev/null 2>&1
!rm -rf ~/.cache/matplotlib
import os
os.kill(os.getpid(), 9)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

plt.rc('font', family='NanumGothic')
plt.rcParams['axes.unicode_minus'] = False

plt.figure()
plt.title('한국어 테스트')
plt.show()

In [ ]:
# ============================================================
# STEP 0-2: 라이브러리 설치 + 합성데이터 소스 다운로드
# ============================================================
!pip install -q transformers datasets accelerate scikit-learn matplotlib seaborn pandas
# SmileStyle: 스마일게이트 AI 한국어 17가지 스타일 대화 데이터셋
!wget -q https://raw.githubusercontent.com/smilegate-ai/korean_smile_style_dataset/main/smilestyle_dataset.tsv -O smilestyle_dataset.tsv
# KakaoChatData: 카카오톡 스타일 챗봇 데이터
!wget -q https://raw.githubusercontent.com/Ludobico/KakaoChatData/KakaoChatData.csv -O ChatbotData.csv
# NSMC: 네이버 영화리뷰
!wget -q https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt -O nsmc_train.txt
print("설치 및 다운로드 완료!")

In [ ]:
# ============================================================
# Google Drive 마운트 + 작업 폴더 이동
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import os, shutil
DATA_DIR = '/content/drive/MyDrive/DLthon'
os.chdir(DATA_DIR)

# 다운받은 파일들을 Drive 폴더로 복사 (참조 편의)
for fname in ['smilestyle_dataset.tsv', 'ChatbotData.csv', 'nsmc_train.txt']:
    src = f'/content/{fname}'
    if os.path.exists(src) and not os.path.exists(fname):
        shutil.copy2(src, fname)

for f in ['train.csv', 'test.csv', 'submission.csv']:
    if os.path.exists(f):
        print(f"  [OK] {f}")
    else:
        print(f"  [!!] {f} 없음")

print(f"\n작업 폴더: {os.getcwd()}")

In [ ]:
import os, random, re, gc, copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from datasets import load_dataset
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

plt.rc('font', family='NanumGothic')
plt.rcParams['axes.unicode_minus'] = False

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ============================================================
# 클래스 매핑 + 하이퍼파라미터
# ============================================================
CLASS_NAMES = ['협박 대화', '갈취 대화', '직장 내 괴롭힘 대화', '기타 괴롭힘 대화', '일반 대화']
CLASS2IDX = {name: idx for idx, name in enumerate(CLASS_NAMES)}
IDX2CLASS = {idx: name for idx, name in enumerate(CLASS_NAMES)}
NUM_CLASSES = 5

MODEL_CONFIGS = [
    {'name': 'beomi/KcELECTRA-base-v2022', 'short': 'KcELECTRA'},
    {'name': 'beomi/kcbert-base',            'short': 'KcBERT'},
]

MAX_LEN = 384
BATCH_SIZE = 16
EPOCHS = 5
LR = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
MAX_GRAD_NORM = 1.0
N_FOLDS = 5
PSEUDO_THRESHOLD = 0.95

# v4 전략
LLRD_FACTOR = 0.95
FGM_EPSILON = 1.0
LABEL_SMOOTHING = 0.05
EMA_DECAY = 0.999

# [v5-3] 예상 test 분포 (간이 분류 기반 - 필요시 조정)
EST_TEST_DIST = {
    0: 39,   # 협박
    1: 23,   # 갈취
    2: 21,   # 직장 내 괴롭힘
    3: 39,   # 기타 괴롭힘
    4: 378,  # 일반 대화
}

print("v5 설정 완료")
print(f"모델: {[m['short'] for m in MODEL_CONFIGS]}")
print(f"K-Fold: {N_FOLDS}, EPOCHS: {EPOCHS}, MAX_LEN: {MAX_LEN}")
print(f"v4: LLRD={LLRD_FACTOR}, FGM={FGM_EPSILON}, LS={LABEL_SMOOTHING}, EMA={EMA_DECAY}")
print(f"v5: EST_TEST_DIST = {EST_TEST_DIST}")

In [ ]:
# ============================================================
# STEP 1: 데이터 로드
# ============================================================
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')
submission_df = pd.read_csv('submission.csv')

print(f"Train: {len(train_df)}개, Test: {len(test_df)}개")
print(f"클래스 분포:\n{train_df['class'].value_counts()}")
print(f"\n>> '일반 대화' 0개 -> STEP 2에서 합성 (v5: 3000+개)")

In [ ]:
# ============================================================
# STEP 2: [v5] 합성 일반대화 (개별 문장 기반, 3000+개)
# ============================================================
# v2/v3: 랜덤 문장 3~5개 이어붙이기 -> 부자연스러운 합성 데이터
# v5: 개별 문장 그대로 사용 -> 자연스러운 일반 대화
# ============================================================
THREAT_KEYWORDS = [
    '죽여', '죽일', '찔러', '칼로', '패줄', '두들겨', '불질러',
    '협박', '신고', '경찰', '감옥', '고소', '소송',
    '돈 내놔', '송금', '이자', '빚', '갚아',
    '해고', '짤리', '사직서', '퇴사', '상사',
    '따돌', '왕따', '무시', '괴롭'
]

def contains_threat(text):
    return any(kw in str(text) for kw in THREAT_KEYWORDS)

normal_samples = []

# ── 소스 1: SmileStyle 개별 문장 (1200개) ──
# [v5-1] 이어붙이기 대신 개별 문장 그대로 사용
print("소스 1: SmileStyle (개별 문장)")
try:
    smile_df = pd.read_csv('smilestyle_dataset.tsv', sep='\t')
    target_cols = [c for c in smile_df.columns
                   if any(kw in c.lower() for kw in ['informal', 'chat', '반말', 'casual'])]
    if not target_cols:
        target_cols = smile_df.columns[1:].tolist()
    smile_texts = []
    for col in target_cols:
        smile_texts.extend(smile_df[col].dropna().tolist())
    # 개별 문장 필터링 (이어붙이기 X)
    smile_filtered = [str(t).strip() for t in smile_texts
                      if not contains_threat(t) and 15 < len(str(t).strip()) < 300]
    random.shuffle(smile_filtered)
    normal_samples.extend(smile_filtered[:1200])
    print(f"  -> {min(1200, len(smile_filtered))}개 수집")
except Exception as e:
    print(f"  오류: {e}")

# ── 소스 2: kor_unsmile 클린 개별 문장 (800개) ──
print("소스 2: kor_unsmile (개별 문장)")
try:
    unsmile_ds = load_dataset('smilegate-ai/kor_unsmile', split='train')
    unsmile_df = unsmile_ds.to_pandas()
    if 'clean' in unsmile_df.columns:
        clean_texts = unsmile_df[unsmile_df['clean'] == 1]['문장'].tolist()
    else:
        label_cols = [c for c in unsmile_df.columns if c not in ['문장', 'clean']]
        clean_mask = unsmile_df[label_cols].sum(axis=1) == 0
        clean_texts = unsmile_df[clean_mask]['문장'].tolist()
    clean_filtered = [str(t).strip() for t in clean_texts
                      if not contains_threat(t) and 10 < len(str(t).strip()) < 300]
    random.shuffle(clean_filtered)
    normal_samples.extend(clean_filtered[:800])
    print(f"  -> {min(800, len(clean_filtered))}개 수집")
except Exception as e:
    print(f"  오류: {e}")

# ── 소스 3: KakaoChatData Q+A 쌍 (500개) ──
# [v5-1] Q+A를 자연스러운 대화 쌍으로 사용
print("소스 3: KakaoChatData (Q+A 쌍)")
try:
    kakao_df = pd.read_csv('ChatbotData.csv')
    kakao_pairs = []
    for _, row in kakao_df.iterrows():
        q = str(row.get('Q', row.iloc[0]))
        a = str(row.get('A', row.iloc[1]))
        conv = f"{q} {a}"
        if not contains_threat(conv) and 20 < len(conv) < 400:
            kakao_pairs.append(conv)
    random.shuffle(kakao_pairs)
    normal_samples.extend(kakao_pairs[:500])
    print(f"  -> {min(500, len(kakao_pairs))}개 수집")
except Exception as e:
    print(f"  오류: {e} (건너뜀)")

# ── 소스 4: NSMC 긍정 리뷰 개별 문장 (500개) ──
print("소스 4: NSMC (개별 문장)")
try:
    nsmc_df = pd.read_csv('nsmc_train.txt', sep='\t')
    positive = nsmc_df[nsmc_df['label'] == 1]['document'].dropna().tolist()
    pos_filtered = [str(t).strip() for t in positive
                    if not contains_threat(t) and 15 < len(str(t).strip()) < 200]
    random.shuffle(pos_filtered)
    normal_samples.extend(pos_filtered[:500])
    print(f"  -> {min(500, len(pos_filtered))}개 수집")
except Exception as e:
    print(f"  오류: {e}")

# ── 소스 5: [v5-5] 템플릿 일상 대화 (100개) ──
print("소스 5: 템플릿 일상 대화")
template_conversations = [
    # 인사/안부
    "안녕 오랜만이다 잘 지냈어",
    "요즘 어떻게 지내 바쁘지",
    "오늘 하루 어땠어 나는 좋았어",
    "잘 자 내일 보자 좋은 꿈 꿔",
    "굿모닝 오늘도 화이팅 하자",
    "야 잘 지내지 연락 좀 해라 보고 싶다",
    "뭐 해 심심한데 놀자",
    "나 왔어 지금 어디야 빨리 와",
    "수고했어 오늘 진짜 고생 많았다",
    "생일 축하해 선물 뭐 갖고 싶어 말해",
    "새해 복 많이 받아 올해도 잘 부탁해",
    "주말 잘 보내 푹 쉬어라",
    "감기 조심해 요즘 환절기라 아픈 사람 많다",
    "졸업 축하한다 진짜 대단해 고생했어",
    "집에 잘 들어갔어 조심해서 가 연락해",
    # 음식/식사
    "점심 뭐 먹을까 나 진짜 배고파",
    "치킨 시킬까 피자 시킬까 고민된다",
    "오늘 저녁은 삼겹살 어때 고기 먹고 싶다",
    "카페 갈래 커피 한잔 하고 싶어",
    "이 맛집 진짜 맛있대 같이 가보자",
    "라면 끓여 먹을까 귀찮은데 맛있겠다",
    "배달 시키자 밖에 나가기 싫어",
    "아이스 아메리카노 사다 줘 더워서 시원한 거 마시고 싶어",
    "떡볶이 먹고 싶다 매운 거 땡긴다",
    "밥 먹었어 아직이면 같이 먹으러 가자",
    "디저트 뭐 먹을까 케이크 어때",
    "맥주 한잔 하자 퇴근하고 치맥 어때",
    # 취미/여가
    "영화 뭐 볼까 요즘 뭐 재밌는 거 있어",
    "게임 하자 같이 한판 할래",
    "넷플릭스 뭐 볼 거 있어 추천 좀 해줘",
    "운동 같이 갈래 헬스장 가자",
    "책 읽고 있는데 이거 진짜 재밌다",
    "노래 듣고 있어 이 노래 너무 좋다 들어봐",
    "여행 가고 싶다 어디로 갈까 바다 어때",
    "주말에 등산 갈래 날씨 좋다고 하더라",
    "콘서트 표 샀다 같이 갈 사람 구한다",
    "자전거 타러 갈래 한강 가자",
    "요가 시작했는데 생각보다 어렵다 재밌어",
    "캠핑 가고 싶다 텐트 준비해야 하는데",
    "사진 찍으러 가자 날씨 좋을 때",
    # 날씨/계절
    "오늘 날씨 좋다 어디 나들이 가자",
    "비 온대 우산 꼭 챙겨",
    "너무 덥다 아이스크림 먹자",
    "눈 온다 밖에 나가서 눈사람 만들까",
    "바람 많이 분다 따뜻하게 입어",
    "봄이다 벚꽃 보러 갈까 여의도 어때",
    "가을이라 단풍 예쁘다 사진 찍자",
    "미세먼지 심하다 마스크 쓰고 다녀",
    # 쇼핑/물건
    "이거 사고 싶다 예쁘지 않아 어때",
    "세일 한다 같이 쇼핑 가자",
    "택배 왔다 드디어 도착 빨리 열어보자",
    "핸드폰 바꿀까 새로 나온 거 괜찮아 보여",
    "옷 사러 가자 봄옷 필요해",
    "인터넷으로 주문했어 내일 온대 기대된다",
    "이거 선물로 괜찮을까 친구 생일인데",
    "신발 추천해줘 운동화 사고 싶어",
    # 학교/공부
    "숙제 다 했어 나 아직 안 했는데",
    "시험 언제야 공부 해야 하는데 하기 싫다",
    "오늘 수업 재밌었어 선생님이 재밌게 해줬어",
    "도서관 같이 갈래 공부하자",
    "발표 준비 해야 돼 좀 긴장된다",
    "성적 나왔어 생각보다 잘 나와서 다행이다",
    "동아리 활동 재밌어 너도 같이 하자",
    "레포트 써야 하는데 주제 뭐로 할까",
    # 감정/기분
    "오늘 기분 좋다 좋은 일이 있었거든",
    "좀 피곤해 어제 늦게 잤어",
    "기분 전환하러 밖에 나가자 바람 쐬러",
    "스트레스 받아 맛있는 거 먹으러 가자",
    "행복하다 오늘 진짜 좋은 하루였어",
    "설렌다 내일 기대되는 일이 있어",
    "보고 싶어 빨리 만나자 언제 시간 돼",
    "진짜 웃긴다 이것 좀 봐",
    "감동이야 정말 고마워 최고다",
    "뿌듯하다 열심히 한 보람이 있네",
    # 약속/계획
    "이번 주말에 만나자 어디서 볼까",
    "몇 시에 만날까 장소는 어디로 할까",
    "내일 약속 있어서 못 만나 미안 다음에 보자",
    "다음 주에 밥 먹자 시간 맞춰보자",
    "여행 계획 세우자 어디로 갈까 제주도 어때",
    "생일 파티 할 건데 올 수 있어",
    "퇴근하고 만나자 6시쯤 어때",
    "영화 예매했어 7시 거 괜찮지",
    "주말에 집들이 하는데 놀러 와",
    "방학 때 뭐 할 거야 같이 뭐 하자",
    # 가족
    "엄마가 밥 하셨대 집에 가서 먹자",
    "동생이랑 싸웠어 짜증나 근데 곧 풀릴 거야",
    "가족 여행 간다 다음 주에 부산 가",
    "아버지 생신이야 선물 뭐 살까",
    "할머니 댁에 가야 해 주말에 같이 가자",
    "강아지가 너무 귀여워 사진 보여줄까",
    "고양이가 또 장난쳤어 너무 웃겨",
    "가족들이랑 외식했어 맛있었다",
    # 일상
    "오늘 일찍 일어났어 개운하다",
    "버스 놓쳤어 다음 거 타야겠다",
    "알람을 안 듣고 늦잠 잤어 큰일이다",
    "오늘 청소했더니 깨끗해져서 기분 좋다",
    "빨래 널어야 하는데 비 온대 어쩌지",
    "머리 잘랐어 어때 괜찮아 보여",
    "새로운 카페 발견했어 분위기 좋더라",
    "오늘 요리 해봤는데 의외로 잘 됐어",
]
normal_samples.extend(template_conversations)
print(f"  -> {len(template_conversations)}개 추가")

# ── 소스 6: 경계 케이스 (50개, test 분포 기반) ──
print("소스 6: 경계 케이스 (test 분포 기반)")
boundary_cases = [
    # 패턴 A: 협박처럼 보이지만 일반 대화 (15개)
    "야 죽을래 ㅋㅋ 아 진짜 웃겨서 죽겠다 아 배아파 ㅋㅋㅋ 진짜 미쳤어 너 개그맨 해라",
    "야 너 진짜 맞을래 ㅋㅋ 아 왜 그런 말을 해서 웃기게 만들어 아 진짜 복근 생기겠다",
    "죽여버린다 ㅋㅋ 아 이 게임 왜 이렇게 어려워 보스 죽여버리고 싶다 진짜",
    "패버리고 싶다 ㅋㅋ 누구를 이 게임 캐릭터 진짜 짜증나 아 다시 해야지",
    "칼로 자르고 싶다 뭘 이 케이크 너무 예뻐서 자르기 아까운데 먹어야지",
    "야 한대 맞을래 ㅋㅋㅋ 농담이야 근데 진짜 왜 그런 얘기를 해 아 웃겨",
    "미쳤어 진짜 ㅋㅋ 이 짤 봤어 와 진짜 웃겨서 죽는줄 알았어 보내줄까",
    "너 진짜 미쳤다 ㅋㅋ 이걸 어떻게 생각해내 와 천재 아니야 대단하다 진짜",
    "야 너 오늘 죽었다 ㅋㅋ 왜 노래방에서 내 노래 뺏어서 불렀잖아 다음엔 내가 먼저",
    "때려치우고 싶다 뭘 회사 오늘 진짜 힘들었어 야 치킨 먹자 나 오늘 자격 있어",
    "경찰 부를거야 ㅋㅋ 왜 너 방 너무 더러워서 환경오염 신고해야 할 것 같아",
    "고소할거야 진짜 ㅋㅋ 왜 네가 나한테 이렇게 맛있는 걸 안 알려줬어 배신자",
    "불질러버리고 싶다 뭘 이 과제 너무 많아서 다 태워버리고 싶어 ㅋㅋ 농담이야",
    "찔러버린다 ㅋㅋ 뭘 이 젤리 포크로 찔러서 먹어야 하는데 손으로 먹었어",
    "감옥에 넣어야 해 ㅋㅋ 누구를 이렇게 늦게 연락하는 너를 지각죄로 가둬야지",
    # 패턴 B: 기타 괴롭힘처럼 보이지만 일반 대화 (15개)
    "야 너 왜 혼자 밥 먹어 같이 먹자 아 몰랐어 미안 내일부터 같이 가자",
    "쟤 좀 이상하지 않아 뭐가 아 그냥 오늘 옷 되게 특이하게 입었더라 멋있어",
    "왕따 당하는 기분이야 ㅋㅋ 왜 아 오늘 점심 메뉴 나만 몰랐어 다들 알려줘",
    "무시하지 마 ㅋㅋ 내 말 좀 들어봐 이번 여행 계획 진짜 좋거든 어디냐면",
    "따돌리는거야 ㅋㅋ 왜 단톡방에 나만 안 넣었어 아 새로 만든거야 초대해줘",
    "너네 나 없이 놀았지 ㅋㅋ 사진 봤어 아 미안 갑자기 된거야 다음엔 꼭 같이 가자",
    "괴롭히지 마 ㅋㅋ 야 자꾸 내 별명 부르지 마 그거 초등학교 때 거잖아 창피해",
    "소외감 느껴 ㅋㅋ 왜 너네 다 커플이라 나만 혼자야 아 나도 소개팅 시켜줘",
    "야 쟤 왜 저래 아 그냥 원래 좀 조용한 애야 착해 알고 보면 진짜 재밌어",
    "나만 빼고 다 아는거야 뭘 아 그 유행어 나만 모르나 알려줘 뭔데",
    "혼자 있고 싶어 왜 아 그냥 오늘 좀 피곤해서 내일 만나자 그래 푹 쉬어",
    "무시당한 기분이야 ㅋㅋ 왜 아 내가 추천한 맛집을 아무도 안 가줘서 나만 좋아하나",
    "같이 안 놀아줘 ㅋㅋ 누가 우리 강아지가 다른 강아지 싫어해서 혼자 놀아 귀여워",
    "차별하는거 아니야 ㅋㅋ 왜 아 치킨 양념만 시키지 말고 후라이드도 시키자 나는 후라이드파",
    "아무도 안 좋아해 ㅋㅋ 뭘 이 맛집을 아무도 모르더라 나만 알고 있어 데려갈까",
    # 패턴 C: 갈취처럼 보이지만 일반 대화 (10개)
    "돈 내놔 ㅋㅋ 밥값 네가 쏜다며 아 맞다 내가 쏜다고 했지 ㅋㅋ 어디 갈까",
    "야 천원만 빌려줘 자판기 커피 마시고 싶은데 지갑을 놓고 왔어 내일 바로 갚을게",
    "야 그거 빌려줘 뭘 충전기 배터리 없어서 잠깐만 쓸게 고마워",
    "밥 사라 ㅋㅋ 야 오늘 내 생일인데 당연히 네가 사야지 어디 갈까",
    "용돈 다 썼어 ㅋㅋ 야 오늘 커피 한잔만 사줘 다음에 내가 쏠게 진짜로",
    "야 택시비 좀 보태줘 3천원만 있으면 되는데 집에 가야 해 내일 바로 보내줄게",
    "야 이거 줘 뭐 그 펜 좀 잠깐만 쓸게 아 고마워 나중에 돌려줄게",
    "돈 있어 얼마 만원만 있으면 되는데 같이 밥 먹으러 가자 내가 부족한 부분 낼게",
    "야 담배 한 개비 줘봐 아 나 오늘 스트레스 받아서 한 대만 ㅋㅋ 고마워 내일 사줄게",
    "이거 나 좀 줘 뭐 이 과자 맛있어 보여서 하나만 줘봐 오 진짜 맛있다",
    # 패턴 D: 직장 괴롭힘처럼 보이지만 일반 대화 (10개)
    "오늘 야근이야 또 아 진짜 힘들다 그래도 이번 프로젝트 끝나면 좀 쉴 수 있겠지",
    "상사가 또 일 줬어 근데 뭐 그래도 인정해주니까 열심히 해야지 파이팅",
    "퇴사하고 싶다 ㅋㅋ 아 농담이야 월급날이니까 참는거지 오늘 뭐 먹을까",
    "야 우리 부장님 또 회식 잡았대 아 귀찮다 그래도 고기니까 ㅋㅋ 가자",
    "아 오늘 진짜 일 많다 죽겠어 ㅋㅋ 그래도 퇴근하면 치맥이다 버텨보자",
    "팀장님이 또 수정해달래 세번째야 근데 뭐 덕분에 더 좋아지긴 했어 감사하지",
    "인사팀에서 면담하자고 했어 뭐래 아 그냥 만족도 조사래 놀랐잖아 ㅋㅋ",
    "해고당할뻔 했어 ㅋㅋ 왜 아 늦잠자서 지각했는데 부장님이 웃으면서 넘어가줬어 휴",
    "사직서 쓸뻔 했다 ㅋㅋ 왜 프린터가 안 돼서 30분 싸웠어 결국 고쳤어",
    "회의 또 해 진짜 오늘만 세번째야 그래도 뭐 좋은 아이디어 나왔으니까 괜찮아",
]
normal_samples.extend(boundary_cases)
print(f"  -> {len(boundary_cases)}개 추가")
print(f"    협박 경계 15, 기타괴롭힘 경계 15, 갈취 경계 10, 직장괴롭힘 경계 10")
print(f"\n총 합성 일반대화: {len(normal_samples)}개")
print(f"  [v5-2] 목표: 3000+ (v2의 1000 대비 3배 이상)")

In [ ]:
# ============================================================
# 합성 데이터 통합 + 전처리
# ============================================================
normal_df = pd.DataFrame({
    'idx': [f'n_{i:03d}' for i in range(len(normal_samples))],
    'class': '일반 대화',
    'conversation': normal_samples
})

train_full = pd.concat([train_df[['idx', 'class', 'conversation']], normal_df],
                       ignore_index=True)
print(f"통합 train: {len(train_full)}개")
print(train_full['class'].value_counts())

def preprocess(text):
    text = str(text)
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^가-힣a-zA-Z0-9ㄱ-ㅎㅏ-ㅣ\s,.!?~ㅋㅎㅠㅜ]', '', text)
    return text.strip()

train_full['conversation'] = train_full['conversation'].apply(preprocess)
test_df['conversation'] = test_df['conversation'].apply(preprocess)
train_full['label'] = train_full['class'].map(CLASS2IDX).astype(int)

print(f"\n전처리 완료. 라벨 분포:")
label_dist = train_full['label'].value_counts().sort_index()
print(label_dist)

# [v5-3] train prior 계산 (calibration에 사용)
train_prior = (label_dist / label_dist.sum()).values
print(f"\ntrain prior: {dict(zip(CLASS_NAMES, [f'{p:.3f}' for p in train_prior]))}")

total_test = sum(EST_TEST_DIST.values())
test_prior = np.array([EST_TEST_DIST[i] / total_test for i in range(NUM_CLASSES)])
print(f"test prior (추정): {dict(zip(CLASS_NAMES, [f'{p:.3f}' for p in test_prior]))}")

# calibration 비율 미리 계산
cal_ratio = test_prior / (train_prior + 1e-8)
print(f"\ncalibration ratio: {dict(zip(CLASS_NAMES, [f'{r:.2f}' for r in cal_ratio]))}")

In [ ]:
# ============================================================
# STEP 4: Dataset, Loss, FGM, EMA, LLRD
# ============================================================
class DKTCDataset(Dataset):
    def __init__(self, texts, labels=None, tokenizer=None, max_len=256):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            str(self.texts[idx]),
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        item = {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
        }
        if 'token_type_ids' in encoding:
            item['token_type_ids'] = encoding['token_type_ids'].squeeze(0)
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item


class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, label_smoothing=0.0, reduction='mean'):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.label_smoothing = label_smoothing
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(
            inputs, targets, weight=self.alpha,
            reduction='none', label_smoothing=self.label_smoothing
        )
        pt = torch.exp(-F.cross_entropy(inputs, targets, reduction='none'))
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        if self.reduction == 'mean':
            return focal_loss.mean()
        return focal_loss


def compute_rdrop_loss(logits1, logits2, labels, loss_fn, alpha=0.7):
    loss1 = loss_fn(logits1, labels)
    loss2 = loss_fn(logits2, labels)
    ce_loss = (loss1 + loss2) / 2
    p = F.log_softmax(logits1, dim=-1)
    q = F.log_softmax(logits2, dim=-1)
    kl_loss = (
        F.kl_div(p, q.exp(), reduction='batchmean') +
        F.kl_div(q, p.exp(), reduction='batchmean')
    ) / 2
    return ce_loss + alpha * kl_loss


class FGM:
    # [v4-2] Fast Gradient Method
    def __init__(self, model, epsilon=1.0, emb_name='word_embeddings'):
        self.model = model
        self.epsilon = epsilon
        self.emb_name = emb_name
        self.backup = {}

    def attack(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad and self.emb_name in name:
                self.backup[name] = param.data.clone()
                if param.grad is not None:
                    norm = torch.norm(param.grad)
                    if norm != 0 and not torch.isnan(norm):
                        r_at = self.epsilon * param.grad / norm
                        param.data.add_(r_at)

    def restore(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad and self.emb_name in name:
                if name in self.backup:
                    param.data = self.backup[name]
        self.backup = {}


class EMA:
    # [v4-4] Exponential Moving Average
    def __init__(self, model, decay=0.999):
        self.model = model
        self.decay = decay
        self.shadow = {}
        self.backup = {}
        for name, param in model.named_parameters():
            if param.requires_grad:
                self.shadow[name] = param.data.clone()

    def update(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                new_avg = (1 - self.decay) * param.data + self.decay * self.shadow[name]
                self.shadow[name] = new_avg.clone()

    def apply_shadow(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                self.backup[name] = param.data.clone()
                param.data = self.shadow[name]

    def restore(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                param.data = self.backup[name]
        self.backup = {}


def get_llrd_optimizer(model, lr=2e-5, weight_decay=0.01, llrd_factor=0.95):
    # [v4-1] Layer-wise Learning Rate Decay
    no_decay = ['bias', 'LayerNorm.weight', 'LayerNorm.bias']

    if hasattr(model, 'electra'):
        backbone = model.electra
    elif hasattr(model, 'bert'):
        backbone = model.bert
    else:
        return torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    layers = [backbone.embeddings] + list(backbone.encoder.layer)
    num_layers = len(layers)

    opt_params = []
    for i, layer in enumerate(layers):
        layer_lr = lr * (llrd_factor ** (num_layers - 1 - i))
        opt_params.extend([
            {'params': [p for n, p in layer.named_parameters()
                       if not any(nd in n for nd in no_decay) and p.requires_grad],
             'lr': layer_lr, 'weight_decay': weight_decay},
            {'params': [p for n, p in layer.named_parameters()
                       if any(nd in n for nd in no_decay) and p.requires_grad],
             'lr': layer_lr, 'weight_decay': 0.0},
        ])

    cls_decay, cls_no_decay = [], []
    for name, param in model.named_parameters():
        if 'classifier' in name and param.requires_grad:
            if any(nd in name for nd in no_decay):
                cls_no_decay.append(param)
            else:
                cls_decay.append(param)
    if cls_decay:
        opt_params.append({'params': cls_decay, 'lr': lr, 'weight_decay': weight_decay})
    if cls_no_decay:
        opt_params.append({'params': cls_no_decay, 'lr': lr, 'weight_decay': 0.0})

    return torch.optim.AdamW(opt_params)

print("v5 클래스 정의 완료")

In [ ]:
# ============================================================
# STEP 5: 학습/검증/예측 함수
# ============================================================
def train_one_epoch(model, dataloader, optimizer, scheduler, loss_fn,
                    fgm=None, ema=None, use_rdrop=True, rdrop_alpha=0.7):
    model.train()
    total_loss = 0
    all_preds, all_labels = [], []

    for batch in dataloader:
        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels = batch['labels'].to(DEVICE)
        model_kwargs = {'input_ids': input_ids, 'attention_mask': attention_mask}
        if 'token_type_ids' in batch:
            model_kwargs['token_type_ids'] = batch['token_type_ids'].to(DEVICE)

        if use_rdrop:
            outputs1 = model(**model_kwargs)
            outputs2 = model(**model_kwargs)
            loss = compute_rdrop_loss(
                outputs1.logits, outputs2.logits, labels,
                loss_fn, alpha=rdrop_alpha
            )
            logits = outputs1.logits
        else:
            outputs = model(**model_kwargs)
            logits = outputs.logits
            loss = loss_fn(logits, labels)

        optimizer.zero_grad()
        loss.backward()

        # [v4-2] FGM
        if fgm is not None:
            fgm.attack()
            adv_out = model(**model_kwargs)
            adv_loss = loss_fn(adv_out.logits, labels)
            adv_loss.backward()
            fgm.restore()

        torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
        optimizer.step()
        scheduler.step()

        if ema is not None:
            ema.update()

        total_loss += loss.item()
        preds = torch.argmax(logits, dim=-1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(dataloader)
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='macro')
    return avg_loss, acc, f1


def evaluate(model, dataloader, loss_fn):
    model.eval()
    total_loss = 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels = batch['labels'].to(DEVICE)
            model_kwargs = {'input_ids': input_ids, 'attention_mask': attention_mask}
            if 'token_type_ids' in batch:
                model_kwargs['token_type_ids'] = batch['token_type_ids'].to(DEVICE)

            outputs = model(**model_kwargs)
            loss = loss_fn(outputs.logits, labels)
            total_loss += loss.item()
            preds = torch.argmax(outputs.logits, dim=-1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(dataloader)
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='macro')
    return avg_loss, acc, f1, all_preds, all_labels


def predict_proba(model, dataloader):
    model.eval()
    all_probs = []
    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            model_kwargs = {'input_ids': input_ids, 'attention_mask': attention_mask}
            if 'token_type_ids' in batch:
                model_kwargs['token_type_ids'] = batch['token_type_ids'].to(DEVICE)
            outputs = model(**model_kwargs)
            probs = F.softmax(outputs.logits, dim=-1)
            all_probs.append(probs.cpu().numpy())
    return np.concatenate(all_probs)


def calibrate_probs(probs, train_prior, test_prior):
    # [v5-3] Prior Shift Calibration
    cal_weights = test_prior / (train_prior + 1e-8)
    calibrated = probs * cal_weights[np.newaxis, :]
    calibrated /= calibrated.sum(axis=1, keepdims=True)
    return calibrated


def optimize_thresholds(oof_probs, oof_labels, num_classes=5):
    best_thresholds = np.ones(num_classes)
    base_preds = np.argmax(oof_probs, axis=1)
    best_f1 = f1_score(oof_labels, base_preds, average='macro')
    print(f"  기본 OOF F1: {best_f1:.4f}")

    for cls in range(num_classes):
        cls_best_t = 1.0
        for t in np.arange(0.3, 3.01, 0.05):
            adjusted = oof_probs.copy()
            adjusted[:, cls] *= t
            preds = np.argmax(adjusted, axis=1)
            f1 = f1_score(oof_labels, preds, average='macro')
            if f1 > best_f1:
                best_f1 = f1
                cls_best_t = t
        best_thresholds[cls] = cls_best_t

    print(f"  최적화 후 OOF F1: {best_f1:.4f}")
    print(f"  Thresholds: {dict(zip(CLASS_NAMES, [f'{t:.2f}' for t in best_thresholds]))}")
    return best_thresholds

print("v5 학습/평가/예측/보정 함수 정의 완료")

In [ ]:
# ============================================================
# STEP 6: K-Fold x Multi-Model 학습
# ============================================================
all_model_probs = []
all_fold_results = []
all_oof_probs = []
oof_labels = train_full['label'].values.copy()

for model_cfg in MODEL_CONFIGS:
    model_name = model_cfg['name']
    model_short = model_cfg['short']
    print(f"\n{'='*60}")
    print(f"  모델: {model_short} ({model_name})")
    print(f"{'='*60}")

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    test_dataset = DKTCDataset(
        test_df['conversation'].values, labels=None,
        tokenizer=tokenizer, max_len=MAX_LEN
    )
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    model_oof = np.zeros((len(train_full), NUM_CLASSES))

    for fold, (train_idx, val_idx) in enumerate(skf.split(train_full, train_full['label'])):
        print(f"\n  --- {model_short} Fold {fold+1}/{N_FOLDS} ---")

        fold_train = train_full.iloc[train_idx]
        fold_val = train_full.iloc[val_idx]

        train_dataset = DKTCDataset(
            fold_train['conversation'].values, fold_train['label'].values,
            tokenizer, MAX_LEN
        )
        val_dataset = DKTCDataset(
            fold_val['conversation'].values, fold_val['label'].values,
            tokenizer, MAX_LEN
        )
        train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

        label_counts = fold_train['label'].value_counts().sort_index()
        total = len(fold_train)
        class_weights = torch.tensor(
            [total / (NUM_CLASSES * count) for count in label_counts.values],
            dtype=torch.float32
        ).to(DEVICE)

        loss_fn = FocalLoss(
            alpha=class_weights, gamma=2.0,
            label_smoothing=LABEL_SMOOTHING
        ).to(DEVICE)

        model = AutoModelForSequenceClassification.from_pretrained(
            model_name, num_labels=NUM_CLASSES
        ).to(DEVICE)

        optimizer = get_llrd_optimizer(
            model, lr=LR, weight_decay=WEIGHT_DECAY, llrd_factor=LLRD_FACTOR
        )
        total_steps = len(train_loader) * EPOCHS
        warmup_steps = int(total_steps * WARMUP_RATIO)
        scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

        fgm = FGM(model, epsilon=FGM_EPSILON)
        ema = EMA(model, decay=EMA_DECAY)

        best_val_f1 = 0
        best_state = None

        for epoch in range(EPOCHS):
            train_loss, train_acc, train_f1 = train_one_epoch(
                model, train_loader, optimizer, scheduler, loss_fn,
                fgm=fgm, ema=ema, use_rdrop=True, rdrop_alpha=0.7
            )
            ema.apply_shadow()
            val_loss, val_acc, val_f1, _, _ = evaluate(model, val_loader, loss_fn)
            if val_f1 > best_val_f1:
                best_val_f1 = val_f1
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            ema.restore()

            print(f"    Ep {epoch+1}/{EPOCHS} | "
                  f"Train F1: {train_f1:.4f} | Val F1: {val_f1:.4f}"
                  f"{'  *' if val_f1 >= best_val_f1 else ''}")

        print(f"    >> Best Val F1: {best_val_f1:.4f}")

        model.load_state_dict(best_state)
        model.to(DEVICE)

        # Test predictions
        fold_probs = predict_proba(model, test_loader)
        all_model_probs.append(fold_probs)

        # OOF predictions
        val_probs = predict_proba(model, val_loader)
        model_oof[val_idx] = val_probs

        all_fold_results.append({
            'model': model_short,
            'fold': fold + 1,
            'best_val_f1': best_val_f1
        })

        del model, fgm, ema
        gc.collect()
        torch.cuda.empty_cache()

    all_oof_probs.append(model_oof)

print("\n학습 완료!")

In [ ]:
# ============================================================
# STEP 7: [v5-3] Prior Shift Calibration + Threshold Optimization
# ============================================================
print(f"\n{'='*60}")
print(f"  [v5-3] Prior Shift Calibration + Threshold Optimization")
print(f"{'='*60}")

for r in all_fold_results:
    print(f"  {r['model']} Fold {r['fold']}: Val F1 = {r['best_val_f1']:.4f}")

avg_val_f1 = np.mean([r['best_val_f1'] for r in all_fold_results])
print(f"\n  평균 Val F1: {avg_val_f1:.4f}")

# (1) Performance-weighted ensemble
weights = np.array([r['best_val_f1'] for r in all_fold_results])
weights = weights / weights.sum()

ensemble_test = np.zeros_like(all_model_probs[0])
for w, p in zip(weights, all_model_probs):
    ensemble_test += w * p

ensemble_oof = np.mean(all_oof_probs, axis=0)

# 보정 전 예측 확인
raw_preds = np.argmax(ensemble_test, axis=1)
print(f"\n  [보정 전] 예측 분포:")
raw_counts = Counter(raw_preds)
for i in sorted(raw_counts.keys()):
    print(f"    {IDX2CLASS[i]}: {raw_counts[i]}개")

# (2) [v5-3] Prior Shift Calibration
print(f"\n  Prior Shift Calibration 적용:")
print(f"    train prior: {[f'{p:.3f}' for p in train_prior]}")
print(f"    test prior:  {[f'{p:.3f}' for p in test_prior]}")
print(f"    cal ratio:   {[f'{r:.2f}' for r in cal_ratio]}")

cal_test = calibrate_probs(ensemble_test, train_prior, test_prior)
cal_oof = calibrate_probs(ensemble_oof, train_prior, test_prior)

cal_preds = np.argmax(cal_test, axis=1)
print(f"\n  [보정 후] 예측 분포:")
cal_counts = Counter(cal_preds)
for i in sorted(cal_counts.keys()):
    print(f"    {IDX2CLASS[i]}: {cal_counts[i]}개")

# (3) Threshold optimization on calibrated OOF
print(f"\n  OOF Threshold 최적화 (보정된 OOF):")
thresholds = optimize_thresholds(cal_oof, oof_labels, NUM_CLASSES)

# (4) Apply thresholds to calibrated test probs
adjusted_test = cal_test.copy()
for cls in range(NUM_CLASSES):
    adjusted_test[:, cls] *= thresholds[cls]

ensemble_preds = np.argmax(adjusted_test, axis=1)
print(f"\n  [최종] 예측 분포 (Calibration + Threshold):")
final_counts = Counter(ensemble_preds)
for i in sorted(final_counts.keys()):
    print(f"    {IDX2CLASS[i]}: {final_counts[i]}개")

In [ ]:
# ============================================================
# STEP 8: Pseudo Labeling + 재학습 (v5 보정 적용)
# ============================================================
print(f"\n{'='*60}")
print(f"  Pseudo Labeling (threshold={PSEUDO_THRESHOLD})")
print(f"{'='*60}")

max_probs = np.max(adjusted_test, axis=1)
confident_mask = max_probs >= PSEUDO_THRESHOLD
pseudo_labels = ensemble_preds[confident_mask]

print(f"  전체 test: {len(test_df)}개")
print(f"  고확신 샘플: {confident_mask.sum()}개 ({confident_mask.sum()/len(test_df)*100:.1f}%)")

if confident_mask.sum() > 50:
    pseudo_counts = Counter(pseudo_labels)
    for i in sorted(pseudo_counts.keys()):
        print(f"    {IDX2CLASS[i]}: {pseudo_counts[i]}개")

    pseudo_df = pd.DataFrame({
        'idx': test_df[confident_mask]['idx'].values,
        'class': [IDX2CLASS[l] for l in pseudo_labels],
        'conversation': test_df[confident_mask]['conversation'].values,
        'label': pseudo_labels
    })
    train_pseudo = pd.concat([train_full, pseudo_df], ignore_index=True)
    print(f"\n  train + pseudo: {len(train_pseudo)}개")

    print(f"  Pseudo 재학습 (KcELECTRA)...")
    tok_p = AutoTokenizer.from_pretrained(MODEL_CONFIGS[0]['name'])

    p_train, p_val = train_test_split(
        train_pseudo, test_size=0.1, stratify=train_pseudo['label'], random_state=SEED
    )
    p_train_ds = DKTCDataset(p_train['conversation'].values, p_train['label'].values, tok_p, MAX_LEN)
    p_val_ds = DKTCDataset(p_val['conversation'].values, p_val['label'].values, tok_p, MAX_LEN)
    p_train_loader = DataLoader(p_train_ds, batch_size=BATCH_SIZE, shuffle=True)
    p_val_loader = DataLoader(p_val_ds, batch_size=BATCH_SIZE)

    p_lc = p_train['label'].value_counts().sort_index()
    p_cw = torch.tensor(
        [len(p_train) / (NUM_CLASSES * c) for c in p_lc.values],
        dtype=torch.float32
    ).to(DEVICE)
    p_loss = FocalLoss(alpha=p_cw, gamma=2.0, label_smoothing=LABEL_SMOOTHING).to(DEVICE)

    p_model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_CONFIGS[0]['name'], num_labels=NUM_CLASSES
    ).to(DEVICE)
    p_opt = get_llrd_optimizer(p_model, lr=LR, weight_decay=WEIGHT_DECAY, llrd_factor=LLRD_FACTOR)
    p_steps = len(p_train_loader) * EPOCHS
    p_sched = get_linear_schedule_with_warmup(p_opt, int(p_steps * WARMUP_RATIO), p_steps)
    p_fgm = FGM(p_model, epsilon=FGM_EPSILON)
    p_ema = EMA(p_model, decay=EMA_DECAY)

    best_pf1 = 0
    best_ps = None

    for epoch in range(EPOCHS):
        tl, ta, tf = train_one_epoch(
            p_model, p_train_loader, p_opt, p_sched, p_loss,
            fgm=p_fgm, ema=p_ema, use_rdrop=True
        )
        p_ema.apply_shadow()
        vl, va, vf, _, _ = evaluate(p_model, p_val_loader, p_loss)
        if vf > best_pf1:
            best_pf1 = vf
            best_ps = {k: v.cpu().clone() for k, v in p_model.state_dict().items()}
        p_ema.restore()

        print(f"    Pseudo Ep {epoch+1}/{EPOCHS} | "
              f"Train F1: {tf:.4f} | Val F1: {vf:.4f}"
              f"{'  *' if vf >= best_pf1 else ''}")

    print(f"    >> Best Pseudo Val F1: {best_pf1:.4f}")

    p_model.load_state_dict(best_ps)
    p_model.to(DEVICE)
    test_ds_p = DKTCDataset(test_df['conversation'].values, labels=None, tokenizer=tok_p, max_len=MAX_LEN)
    test_ld_p = DataLoader(test_ds_p, batch_size=BATCH_SIZE)
    pseudo_probs = predict_proba(p_model, test_ld_p)

    # Calibrate pseudo probs too
    pseudo_probs_cal = calibrate_probs(pseudo_probs, train_prior, test_prior)

    # Final blend (calibrated ensemble + calibrated pseudo)
    final_probs = 0.4 * cal_test + 0.6 * pseudo_probs_cal
    for cls in range(NUM_CLASSES):
        final_probs[:, cls] *= thresholds[cls]
    final_preds = np.argmax(final_probs, axis=1)

    del p_model, p_fgm, p_ema
    gc.collect()
    torch.cuda.empty_cache()
else:
    print("  고확신 샘플 부족 -> Pseudo Labeling 건너뜀")
    final_probs = adjusted_test.copy()
    final_preds = ensemble_preds.copy()

# [v5-4] Confidence-based Normal Fallback
# 위험 클래스(0~3) 중 최대 확률이 매우 낮으면 일반으로 분류
dangerous_max = np.max(final_probs[:, :4], axis=1)
low_conf_mask = dangerous_max < 0.15
if low_conf_mask.sum() > 0:
    print(f"\n  [v5-4] Confidence Fallback: {low_conf_mask.sum()}개 -> 일반 대화")
    final_preds[low_conf_mask] = 4

In [ ]:
# ============================================================
# STEP 9: 제출파일 생성
# ============================================================
submission_df['class'] = final_preds.astype(int)
submission_df.to_csv('submission_v5.csv', index=False)

print("submission_v5.csv 저장 완료!")
print(f"\n최종 예측 분포:")
final_counts = Counter(final_preds)
for i in sorted(final_counts.keys()):
    print(f"  {IDX2CLASS[i]}: {final_counts[i]}개")

normal_count = sum(1 for p in final_preds if p == 4)
print(f"\n일반 대화: {normal_count}개 ({normal_count/len(final_preds)*100:.1f}%)")

# 예상 분포와 비교
print(f"\n예상 vs 실제:")
total_test = sum(EST_TEST_DIST.values())
for i in range(NUM_CLASSES):
    est = EST_TEST_DIST[i]
    act = final_counts.get(i, 0)
    print(f"  {IDX2CLASS[i]}: 예상 {est} | 실제 {act} | 차이 {act - est:+d}")

In [ ]:
# ============================================================
# STEP 10: 결과 시각화
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# (a) Fold별 Val F1
ax = axes[0]
for mc in MODEL_CONFIGS:
    ms = mc['short']
    fold_f1s = [r['best_val_f1'] for r in all_fold_results if r['model'] == ms]
    x_labels = [f"{ms}\nF{i+1}" for i in range(len(fold_f1s))]
    ax.bar(x_labels, fold_f1s, alpha=0.7, label=ms)
ax.set_ylabel('Val F1')
ax.set_title('K-Fold Val F1')
ax.legend()
ax.set_ylim(0.85, 1.0)

# (b) 보정 전 vs 보정 후
ax = axes[1]
labels_list = [IDX2CLASS[i] for i in range(NUM_CLASSES)]
x = np.arange(NUM_CLASSES)
raw_vals = [raw_counts.get(i, 0) for i in range(NUM_CLASSES)]
final_vals = [final_counts.get(i, 0) for i in range(NUM_CLASSES)]
w = 0.35
ax.bar(x - w/2, raw_vals, w, label='보정 전', alpha=0.7)
ax.bar(x + w/2, final_vals, w, label='보정 후 (v5)', alpha=0.7)
ax.set_xticks(x)
ax.set_xticklabels(labels_list, rotation=45, ha='right')
ax.set_title('보정 전 vs 보정 후')
ax.legend()

# (c) 최종 vs 예상 분포
ax = axes[2]
est_vals = [EST_TEST_DIST[i] for i in range(NUM_CLASSES)]
ax.barh(labels_list, final_vals, alpha=0.7, label='v5 예측')
ax.barh(labels_list, est_vals, alpha=0.3, label='예상 분포', edgecolor='red', linewidth=2, fill=False)
ax.set_title('최종 예측 vs 예상 분포')
ax.legend()
for i, v in enumerate(final_vals):
    ax.text(v + 2, i, str(v), va='center')

plt.tight_layout()
plt.savefig('v5_results.png', dpi=150)
plt.show()

In [ ]:
# ============================================================
# STEP 11: 프로젝트 정리
# ============================================================
print("=" * 60)
print("  DKTC v5 프로젝트 정리")
print("=" * 60)
print(f"\n  [v3] K-Fold: {N_FOLDS}F x {len(MODEL_CONFIGS)} models, R-Drop, Focal Loss")
print(f"  [v3] Pseudo Labeling: threshold={PSEUDO_THRESHOLD}")
print(f"  [v3] MAX_LEN: {MAX_LEN}")
print(f"\n  [v4] LLRD={LLRD_FACTOR}, FGM={FGM_EPSILON}, EMA={EMA_DECAY}, LS={LABEL_SMOOTHING}")
print(f"\n  [v5-1] Natural Individual Sampling (개별 문장 사용)")
print(f"  [v5-2] 3x Normal Scale-up: {len(normal_samples)}개 합성 일반대화")
print(f"  [v5-3] Prior Shift Calibration: {[f'{r:.2f}' for r in cal_ratio]}")
print(f"  [v5-4] Confidence Fallback: 위험 확률 < 0.15 -> 일반")
print(f"  [v5-5] Template Augmentation: {len(template_conversations)}개 수작업 대화")
print(f"\n  평균 Val F1: {avg_val_f1:.4f}")
print(f"  제출파일: submission_v5.csv")
print(f"\n  완료!")

In [ ]:
from google.colab import files
files.download('submission_v5.csv')